<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# CP101: Lab 5 - Dictionaries, Pandas I, and the American Community Survey

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("CP101_LAB05.ipynb")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## Learning Objectives

By the end of this lab, you will be able to:

1. define the **Decennial Census** and the **American Community Survey (ACS)** as data sources,
   compare a count with a sample-based estimate, and say which geographies each product
   publishes;
2. explain why the geographic unit of a row determines what comparisons are meaningful;
3. create, access, and update Python dictionaries and use them in a Pandas workflow;
4. distinguish a Pandas `Series` from a `DataFrame`;
5. inspect, select, filter, sort, and safely modify tabular data;
6. use vectorized numeric operations instead of row-by-row loops;
7. recognize the ACS annotation values and the top code, and say why they pass a null check; and
8. read an ACS variable code and say which table, line and measure it names.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 1. Census and ACS Fundamentals

The U.S. Census Bureau publishes several data products. Two of the most important for
urban analysis are the **Decennial Census** and the **American Community Survey (ACS)**.

| | Decennial Census | American Community Survey |
|---|---|---|
| Frequency | Every 10 years | Conducted continuously; estimates released every year |
| Main purpose | Population and housing counts | Social, economic, housing, and demographic characteristics |
| Count or estimate? | Primarily counts | Sample-based estimates |
| Small-area use | Very detailed geography | ACS 5-year estimates support census tracts and block groups |

The products are not interchangeable.

If your question is simply **how many people live in an area**, a Decennial Census count
may be appropriate.

If your question involves **income, commuting, rent, poverty, disability, educational
attainment, or vehicle access**, you will usually need the ACS.

### ACS 1-year vs. ACS 5-year estimates

ACS estimates are published in different products.

- **1-year estimates** use one year of survey responses and are published only for areas
  of 65,000 people or more.
- **5-year estimates** pool five years of responses and are available for much smaller
  geographies, including census tracts.

For tract-level urban analysis, the **ACS 5-year product** is therefore common. Its year is
the **last** year of the five: the 2023 5-year estimates pool responses from 2019 through
2023 and use 2023 boundaries.

Which geographies each product publishes is the first thing to check before you go looking
for a number:

| Product | Smallest geography published |
| --- | --- |
| Decennial Census | Block |
| ACS 5-year Detailed Tables (the `B` tables, like `B01003`) | Block group |
| ACS 5-year Subject Tables (`S`) and Data Profiles (`DP`) | Census tract |
| ACS 1-year estimates, any table | Areas of 65,000 people or more |

Two consequences. There is no ACS 1-year tract data anywhere, in any table, because no tract
holds 65,000 people. And block-level numbers come only from the decennial census, so a
question about ACS income at the block level has no answer.

But remember: ACS values are **estimates from a sample**, not complete counts of every
resident or household.

Every ACS estimate is published alongside a **margin of error (MOE)**. The simplified
course table in this lab contains estimates only, so we cannot use it to quantify sampling
uncertainty. Lab 7 works with the margins.

### Questions these products can help answer

- How many people live in an area?
- Which tracts have higher median household incomes?
- Where are households without vehicles concentrated?
- How do commuting patterns differ across neighborhoods?
- At what geographic scale can the available data support the question?

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## 2. Why Geography Matters

Geography shapes how the Census Bureau samples and aggregates data. It is not something added
later for mapping: it is embedded in data collection, in the sampling units, and in how the
margins of error are computed. Geographic areas are organized in a hierarchy in which larger
units, like states, include smaller units, like counties and census tracts:

```text
state -> county -> tract -> block group -> block
```

The Bureau draws census tracts to hold between 1,200 and 8,000 people, aiming for about 4,000,
and designs them to be relatively stable over time.

Geography determines **what one row represents**, and therefore what comparisons are
meaningful. A county population supports county-scale comparison; a tract population supports
neighborhood-scale comparison; comparing a county value directly with a tract value mixes
units of analysis, which is a common way to misread Census data.

Lab 6 is about this geography in depth: the codes, how they nest, and how they connect a table
to a map. Today you need two facts.

### The codes are text

Every area has a code. California is state `06` and Alameda County is county `001`. The
leading zeros are part of the code, so these columns must be read and kept as text. We never
calculate their mean or add them together as numbers.

### A note about vintages

Boundary changes happen every decade, and sometimes more often. Tracts may split, merge, or
receive different identifiers, so a tract code is only meaningful together with its
**vintage**, the year of the boundaries it belongs to. Two tables that both contain a tract
identifier are not automatically describing the same piece of ground. Check the vintage.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 3. Load the ACS Table

For this lab, we use a prepared 2023 ACS 5-year tract table for Alameda County.

We are intentionally using a provided file rather than the Census API. Programmatic API
access begins in Lab 6.

In [ ]:
import pandas as pd

acs = pd.read_csv(
    "data/raw/alameda_tracts23.csv",
    dtype={
        "state": "string",
        "county": "string",
        "tract": "string",
    },
)

acs.head()

### Why specify `dtype=` while reading?

`state`, `county`, and `tract` are identifiers.

Reading them as strings preserves leading zeros:

```text
06
001
```

If they were read as integers, `pandas` would turn them into:

```text
6
1
```

The mathematical value is unchanged, but the **identifier is damaged**.

### First inspection

Before transforming a dataset, inspect what you actually loaded.

Useful questions include:

- How many rows and columns are there?
- What are the column names?
- What type is each column?
- What does one row represent?

Useful first checks include `.head()`, `.shape`, `.columns`, `.dtypes`, and `.info()`.
You do not need to memorize all of them at once; the important habit is to inspect the
table before transforming it.


In [ ]:
print("Shape:", acs.shape)

print("\nColumns:")
print(acs.columns.tolist())

print("\nData types:")
display(acs.dtypes)

### What is one row?

In this table, one row represents **one census tract in Alameda County**.

That is the table's **unit of observation**.

Knowing the unit of observation prevents errors later. For example, a column named
`population` means tract population here because each row is a tract.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 4. Python Dictionaries

A dictionary stores information as **key-value pairs**.

```python
dictionary = {
    "key": "value"
}
```

Keys are unique. Dictionaries are mutable, meaning values can be added or updated.

In [ ]:
acs_variables = {
    "population": "Total population estimate",
    "median_household_income": "Median household income estimate",
}

acs_variables

Access a value by its key:

```python
dictionary["key"]
```

Useful dictionary methods include:

- `.keys()` - all keys;
- `.values()` - all values;
- `.items()` - key-value pairs.

In [ ]:
print("Meaning of population:")
print(acs_variables["population"])

print()
print("Keys:", list(acs_variables.keys()))
print("Values:", list(acs_variables.values()))
print("Items:", list(acs_variables.items()))

### Updating a dictionary

You can change an existing value or add a new key.

In [ ]:
acs_variables["NAME"] = "Tract name, as the Bureau prints it"

acs_variables

### ✏️ Try it Out: dictionary practice

Create a dictionary named `geography_codes` with:

- `"state"` mapped to `"06"`
- `"county"` mapped to `"001"`
- `"state_name"` mapped to `"California"`
- `"county_name"` mapped to `"Alameda County"`

Then:

1. print the county code;
2. change `"county_name"` to `"Alameda County, CA"`; and
3. print the updated dictionary.

Keep the codes as text. `"06"` is a label, not the number six.

In [ ]:
geography_codes = ...

print(...)

geography_codes[...] = ...
print(geography_codes)

In [ ]:
grader.check("q1_geography_codes")

### Pro tip: dictionaries can configure Pandas operations

A common example is renaming columns.

In [ ]:
display_names = {
    "population": "population_estimate",
    "median_household_income": "median_income_estimate",
}

renamed_preview = acs.rename(
    columns=display_names
)

renamed_preview[
    [
        "population_estimate",
        "median_income_estimate",
    ]
].head()

Notice that we created `renamed_preview` rather than changing `acs`.

That lets us demonstrate the operation while keeping the original column names for the
rest of the lab.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 5. Pandas Series and DataFrames

A Pandas `DataFrame` is a two-dimensional table.

A Pandas `Series` is a single one-dimensional column.

In [ ]:
population_series = acs["population"]
population_dataframe = acs[["population"]]

print("Single brackets:", type(population_series))
print("List of columns:", type(population_dataframe))

Why does this matter?

Many Pandas methods operate on a Series, while others return or expect a DataFrame.

A useful rule:

```python
df["column"]       # Series
df[["column"]]     # DataFrame
```

### Selecting multiple columns

Pass a list of column names inside the brackets.

> **Pro tip:** `.loc[]` selects by labels/conditions, while `.iloc[]` selects by integer
> position. We will use `.loc[]` more often because analytical conditions are usually
> expressed with labels and Boolean masks.


In [ ]:
core_columns = [
    "NAME",
    "population",
    "median_household_income",
]

acs[core_columns].head()

### Selecting rows and columns with `.loc[]`

`.loc[]` selects using labels and Boolean conditions.

The general pattern is:

```python
df.loc[rows, columns]
```

In [ ]:
acs.loc[
    0:4,
    [
        "NAME",
        "population",
    ],
]

### ✏️ Try it Out: selection

Create a DataFrame named `first_ten_income` containing:

- the first 10 rows; and
- only `NAME` and `median_household_income`.

Use `.loc[]` or `.iloc[]`. Remember that a list of column names inside the brackets returns a DataFrame, while a single name returns a Series.

In [ ]:
first_ten_income = ...
display(first_ten_income)

In [ ]:
grader.check("q2_first_ten_income")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 6. Boolean Filtering

A comparison such as:

```python
acs["population"] >= 5000
```

is evaluated for every row and returns a Boolean Series.

In [ ]:
large_population = (
    acs["population"] >= 5000
)

large_population.head()

`True` means the row satisfies the condition.

`False` means it does not.

A Boolean mask can then be used with `.loc[]`.

In [ ]:
large_tracts = acs.loc[
    large_population,
    [
        "NAME",
        "population",
        "median_household_income",
    ],
].copy()

print("Number of tracts:", len(large_tracts))
display(large_tracts.head())

### Questions filtering can answer

- Which tracts have population above a threshold?
- Which tracts fall within a range?
- Which rows have missing data?
- Which observations satisfy two conditions at once?

Two other useful filtering tools are:

```python
series.between(low, high)
series.isin(["value_a", "value_b"])
```

Use them when they make the condition clearer than several comparisons.

### Combining Boolean conditions

Use:

- `&` for **and**
- `|` for **or**
- `~` for **not**

Put each comparison inside parentheses.

In [ ]:
large_and_high_income = (
    (acs["population"] >= 5000)
    & (acs["median_household_income"] >= 100000)
)

acs.loc[
    large_and_high_income,
    [
        "NAME",
        "population",
        "median_household_income",
    ],
].head()

### ✏️ Try it Out: filtering

Create `lower_income_tracts` containing tracts where:

- median household income is below `$75,000`; **and**
- population is at least `2,000`.

Keep only:

- `NAME`
- `population`
- `median_household_income`

Build each condition as its own named mask first, then combine them with `&` and apply the result with `.loc[]`. How many rows meet both conditions?

In [ ]:
is_lower_income = ...
is_populated = ...

lower_income_tracts = ...
print("Rows meeting both conditions:", len(lower_income_tracts))

In [ ]:
grader.check("q3_lower_income_tracts")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 7. Sorting and Safe Modification

### Sorting

Filtering asks **which rows qualify**. Sorting asks **which rows come first**.

`.sort_values()` returns a new DataFrame ordered by one or more columns.

```python
df.sort_values("column")                     # lowest to highest
df.sort_values("column", ascending=False)    # highest to lowest
df.sort_values(["column_a", "column_b"])     # ties in the first broken by the second
```

Ascending is the default, so lowest-to-highest needs no `ascending=` argument.


In [ ]:
lowest_income_first = large_tracts.sort_values(
    "median_household_income"
)

lowest_income_first.head()

Look at the first row.

`-666666666` is not an income. Sorting did exactly what it was asked to do; the
problem is in the data, and Section 8 is about finding it.

### Questions sorting can answer

- Which tracts have the highest or lowest values?
- What does the top or bottom of the distribution look like?
- Are there values at the extremes that deserve a second look?

> **Pro tip:** `.sort_values("col").head(5)` and `.nsmallest(5, "col")` return the
> same five rows. Reach for `.nsmallest()` / `.nlargest()` when you only want the
> ends of the distribution and do not need the whole table reordered.

### Safe modification: what `.copy()` is for

Section 6 wrote this, and did not explain the last four characters:

```python
large_tracts = acs.loc[mask, columns].copy()
```

`.loc[]` returns a new table built from `acs`. For years, whether a later assignment such as

```python
large_tracts["income_thousands"] = large_tracts["median_household_income"] / 1000
```

could also change `acs` depended on `pandas` internals. Older versions sometimes wrote
through to the original and printed a `SettingWithCopyWarning` that you were expected to
act on.

`pandas` 3, which this course's autograder runs, uses **copy-on-write**: a table derived
from another is always independent, and writing to `large_tracts` cannot change `acs`.
The cell below shows that.

`.copy()` is still worth writing. It states that you intend to modify the result, and it
keeps the code correct and warning-free on any older `pandas` you meet, including some
shared hub environments. The habit is cheap:

> If you select rows or columns and then intend to modify the result, call
> `.copy()` when you select.


In [ ]:
# Write to a selection made WITHOUT .copy(), then check whether acs changed.
selection = acs.loc[acs["population"] >= 5000, ["NAME", "population"]]
selection["population"] = 0

print("Rows in acs with population 0 after the write:", int((acs["population"] == 0).sum()))
print("(2 tracts genuinely have zero population; a larger number would mean acs changed.)")
print()

# The same operation on a copy, which is the habit this lab asks you to keep.
lowest_income_first = lowest_income_first.copy()
lowest_income_first["income_thousands"] = (
    lowest_income_first["median_household_income"] / 1000
)

print("Columns in the copy: ", lowest_income_first.columns.tolist())
print("Columns in acs:      ", acs.columns.tolist())

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 8. Missing and Unavailable ACS Values

A standard first check for missing data is `.isna()`.

In [ ]:
acs[
    [
        "population",
        "median_household_income",
    ]
].isna().sum()

The result suggests there are no missing income values.

But now inspect the minimum.

In [ ]:
print(
    "Minimum median household income:",
    acs["median_household_income"].min(),
)

`-666666666` is not a plausible household income.

In ACS data, this is an **annotation value**: the Bureau's code for an estimate that could
not be computed because there were too few sample observations. The table further down
this section lists the others.

This is a critical data-wrangling lesson:

> Missing or unavailable information is not always represented as `NaN`.

You need both Pandas tools **and knowledge of the source data**.

In [ ]:
unavailable_income = (
    acs["median_household_income"]
    .eq(-666666666)
)

print(
    "Unavailable estimates:",
    unavailable_income.sum(),
)

acs.loc[
    unavailable_income,
    [
        "NAME",
        "population",
        "median_household_income",
    ],
]

Use `.mask()` to replace those coded values with actual missing values.

In [ ]:
acs["median_household_income"] = (
    acs["median_household_income"]
    .mask(unavailable_income)
)

print(
    "Missing income values after cleaning:",
    acs["median_household_income"].isna().sum(),
)

### What happened here?

1. `.eq(-666666666)` compared the entire Series with the special value.
2. The result was a Boolean mask.
3. `.mask()` replaced values where that mask was `True`.
4. We did not write a loop over 379 rows.

That is a **vectorized operation**.

### The Bureau's other codes

`-666666666` is one of six **annotation values** the Bureau uses when it cannot publish a
number. These four are the ones this course meets:

| Value | Which column | What it means |
| --- | --- | --- |
| `-666666666` | the estimate | The estimate could not be computed, usually too few sample observations |
| `-222222222` | the margin of error | The margin of error could not be computed, for the same reason |
| `-333333333` | the margin of error | The estimate is a median that falls in an open-ended top or bottom interval |
| `-555555555` | the margin of error | The estimate is controlled to an independent population estimate, so no margin applies |

Three of the four live in the margin-of-error column, which this file does not carry; you
meet them in Lab 7. All of them pass `.isna()`, which is the point: **a clean null check does
not mean clean data.** Look at the range of your numbers as well.

Two more things are hiding in this column, and neither is a code.

**Top-coding.** Look at the maximum: exactly `250001`. The Bureau does not publish a tract
median above $250,000; data.census.gov shows such a tract as `250,000+`, and the API and the
machine-readable files carry it as `250001`. Every tract at that value could have a true
median of $260,000 or $600,000, and you cannot tell them apart. It is a ceiling, not a
measurement, and averaging this column understates the top of the distribution.

**Zero-population tracts.** A tract can have no residents at all. Regional parks, industrial
land and open water get tract numbers. These are real rows, not errors, but they will distort
any per-capita calculation.

In [ ]:
TOP_CODE = 250001

n_top_coded = (acs["median_household_income"] == TOP_CODE).sum()
n_empty = (acs["population"] == 0).sum()

print("Tracts at the top code:      ", n_top_coded)
print("Tracts with zero population: ", n_empty)
print()
print("The zero-population tracts:")
print(acs.loc[acs["population"] == 0, ["NAME", "population", "median_household_income"]])

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 9. Vectorized Operations

Pandas is designed to perform operations on an entire Series at once.

For example:

```python
acs["population"] / 1000
```

divides every population value by 1,000 without a row-by-row loop.

In [ ]:
acs["population_thousands"] = (
    acs["population"] / 1000
)

acs["income_thousands"] = (
    acs["median_household_income"] / 1000
)

acs["income_above_100k"] = (
    acs["median_household_income"] >= 100000
)

acs[
    [
        "NAME",
        "population_thousands",
        "income_thousands",
        "income_above_100k",
    ]
].head()

One thing to notice in `income_above_100k`. Six tracts now have `NaN` for income, and a
comparison against `NaN` is always `False`. Those six therefore show `False`, which reads
as "not above $100k" when the truth is "unknown". Nothing warns you. Whenever you build a
`True`/`False` column from a column with missing values, decide what the missing rows
should say, and check them.

### Questions vectorized operations can answer

- Which values satisfy a threshold?
- How can a variable be converted to another unit?
- How can we create a new variable from an existing column?
- Which values should be replaced based on a condition?

### Pro tip: prefer vectorized operations to loops

When a Pandas Series operation can express the calculation clearly, it is usually simpler
and more efficient than looping row by row.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## 10. Read an ACS Variable Code

The two numeric columns in this table did not arrive with the names `population` and
`median_household_income`. The Bureau publishes every ACS number under a **variable code**,
and whoever built this file renamed them. When you request data yourself in Lab 6, the codes
are what comes back.

| Code | Table | Line | Measure | The Bureau's label |
| --- | --- | --- | --- | --- |
| `B01003_001E` | `B01003`, Total Population | `001` | `E`, estimate | Estimate!!Total |
| `B19013_001E` | `B19013`, Median Household Income | `001` | `E`, estimate | Estimate!!Median household income in the past 12 months (in 2023 inflation-adjusted dollars) |

Reading a code apart: the letter says which kind of table it is (`B` for a detailed table,
the second row of the products table in Section 1), the five digits identify the table, `_001` is the line within
it, and the final letter is the measure, `E` for the estimate or `M` for its margin of error.
`B19013_001M` is the margin of error for the median income you have been working with.

The label carries two facts you would not otherwise know. The median is over "the past 12
months" as each household answered it, and the dollars are **2023 inflation-adjusted**: a
5-year estimate pools five years of responses and expresses them all in the final year's
dollars. The Bureau publishes every code and label for each year at
[api.census.gov/data/2023/acs/acs5/variables.html](https://api.census.gov/data/2023/acs/acs5/variables.html).

### ✏️ Try it Out: rename by code

The cell below rebuilds the table the way the API delivers it, with the codes as column
names. Write a dictionary `variable_names` that maps each code to the readable name this lab
has used, then pass it to `.rename(columns=...)` to produce `readable`.

Add one more entry to the dictionary: `"B19013_001M"` mapped to `"median_household_income_moe"`.
That column is not in this table. `.rename()` ignores keys it does not find, which is exactly
why one dictionary of names can be written once and reused on every file you pull.

In [ ]:
api_style = acs[["NAME", "population", "median_household_income"]].rename(
    columns={
        "population": "B01003_001E",
        "median_household_income": "B19013_001E",
    }
)

api_style.head()

In [ ]:
variable_names = ...

readable = ...
readable.head()

In [ ]:
grader.check("q5_variable_names")

Three columns came back, not four. `.rename()` silently ignored the `M` key, which here is what
you wanted. It will just as silently ignore a key you misspelled, so after any rename check the
column names by looking, the way Section 3 checked the table before using it.

Section 4 said dictionaries can configure Pandas operations. This is that idea used for real:
one dictionary that documents what each code means and renames the columns at the same time.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 11. Final Practice

Use the cleaned `acs` DataFrame to create a DataFrame named `priority_view`.

It should:

1. keep rows where median household income is available;
2. keep tracts with population at least `4,000`;
3. keep only:
   - `NAME`
   - `population`
   - `median_household_income`
4. create a new column named `income_below_80k`, `True` where median household income is under `$80,000`;
5. sort from **lowest to highest** median household income.

Do not use a loop. Every step here is a vectorized operation.

In [ ]:
priority_view = ...

priority_view["income_below_80k"] = ...

priority_view = ...
display(priority_view.head())

In [ ]:
grader.check("q4_priority_view")

### Check your reasoning

Before moving on, answer in words:

1. What is the unit of observation in `priority_view`?
2. Why were `state`, `county` and `tract` read as text?
3. Why did we replace `-666666666` rather than treat it as a real income?
4. Which operation in your solution was vectorized?
5. If another dataset contains the same tract code but uses 2010 tract boundaries, is a
   direct comparison automatically safe? Why or why not?

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Recap

What this lab covered, and the question each tool answers:

- **Census product**, Decennial Census vs. ACS: which product carries the variable I need, and at which geographies?
- **Geographic scale**, state / county / tract: what does one row represent?
- **Key-value information**, `dict`: how do I document variables, or configure an operation?
- **One column**, `Series`, and **a table**, `DataFrame`: `df["col"]` gives the first, `df[["col"]]` the second.
- **Explicit selection**, `.loc[]` and `.iloc[]`: which rows and columns do I need?
- **Boolean filtering**, comparisons combined with `&`, `|`, `~`, plus `.between()` and `.isin()`: which tracts meet the criteria?
- **Ordering**, `.sort_values()`: which observations are highest or lowest?
- **Missing and coded values**, `.isna()`, `.eq()`, `.mask()`, and a look at the range: is this number a measurement or a code?
- **Whole-column calculation**, vectorized `Series` operations: how do I derive or transform a variable without a loop?
- **Variable codes**, a dictionary passed to `.rename()`: which table, line and measure does this column hold?

### Acknowledgement

This lab adapts material from *MiniLab 5: Census Data in Python* (CYPLAN 101, Summer 2026)
by Li Plass.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

Upload your submission zip to **LAB05 - Dictionaries, Pandas I, and the American Community Survey** on [Gradescope](https://www.gradescope.com/).

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)